# Restaurant Visitor Forecasting

Use this notebook in a private Kaggle workspace with access under the [competition rules](https://www.kaggle.com/competitions/recruit-restaurant-visitor-forecasting/rules). Attach the official competition data using **Add Input**, then run all cells. The loader can read the supplied ZIP files directly. The final cell creates an archive with aggregate error metrics and charts. Keep the competition data and notebook run private.

In [ ]:
"""Load authorized, local competition files without publishing raw records."""

import logging
from io import BytesIO
from pathlib import Path, PurePosixPath
from typing import NamedTuple
from zipfile import BadZipFile, ZipFile, ZipInfo
from zlib import error as DecompressionError

import numpy as np
import pandas as pd

INPUT_FIELDS = {
    "air_visit_data.csv": ("air_store_id", "visit_date", "visitors"),
    "date_info.csv": ("calendar_date", "holiday_flg"),
    "air_reserve.csv": ("air_store_id", "visit_datetime", "reserve_datetime", "reserve_visitors"),
    "sample_submission.csv": ("id", "visitors"),
}
MAX_CSV_BYTES = 128 * 1024**2
MAX_NESTED_ZIP_BYTES = 64 * 1024**2
MAX_ARCHIVE_BYTES = 256 * 1024**2
LOGGER = logging.getLogger(__name__)


class _InputSource(NamedTuple):
    path: Path
    outer_member: ZipInfo | None = None
    inner_member: ZipInfo | None = None


def find_competition_directory(input_root: Path | str) -> Path:
    """Find the sole directory containing all four loose CSVs."""
    root = Path(input_root)
    if not root.is_dir():
        raise FileNotFoundError("The competition input root must be an accessible directory.")
    matches = {
        path.parent
        for path in root.rglob("air_visit_data.csv")
        if all((path.parent / filename).is_file() for filename in INPUT_FIELDS)
    }
    if not matches:
        raise FileNotFoundError("No complete competition CSV directory was found under the input root.")
    if len(matches) > 1:
        raise ValueError("Multiple complete CSV directories were found. Select one competition directory.")
    return matches.pop()


def _member_name(member: ZipInfo) -> str:
    return PurePosixPath(member.filename.replace("\\", "/")).name


def _read_zip_member(archive: ZipFile, member: ZipInfo, limit: int, name: str) -> bytes:
    if member.file_size > limit:
        raise ValueError(f"A ZIP member for {name} exceeds the supported read size.")
    with archive.open(member) as handle:
        content = handle.read(limit + 1)
    if len(content) > limit:
        raise ValueError(f"A ZIP member for {name} exceeds the supported read size.")
    return content


def _add_zip_sources(path: Path, matches: dict[str, list[_InputSource]], *, direct: bool) -> None:
    wrapper = path.name.removesuffix(".zip")
    expected = wrapper if wrapper in INPUT_FIELDS else None
    label = expected or "the competition CSVs"
    try:
        if path.stat().st_size > MAX_ARCHIVE_BYTES:
            if direct or expected:
                raise ValueError(f"A ZIP input for {label} exceeds the supported read size.")
            return
        archive = ZipFile(path)
    except (OSError, BadZipFile):
        if direct or expected:
            raise ValueError(f"Expected a readable ZIP input for {label}.") from None
        return

    with archive:
        for member in archive.infolist():
            if member.is_dir():
                continue
            basename = _member_name(member)
            if basename in INPUT_FIELDS and expected in (None, basename):
                matches[basename].append(_InputSource(path, member))
                continue
            nested_name = basename.removesuffix(".zip")
            if expected or basename == nested_name or nested_name not in INPUT_FIELDS:
                continue
            try:
                packed = _read_zip_member(archive, member, MAX_NESTED_ZIP_BYTES, nested_name)
                with ZipFile(BytesIO(packed)) as inner:
                    for entry in inner.infolist():
                        if not entry.is_dir() and _member_name(entry) == nested_name:
                            matches[nested_name].append(_InputSource(path, member, entry))
            except (BadZipFile, OSError, RuntimeError, NotImplementedError, DecompressionError):
                raise ValueError(f"Expected a readable nested ZIP input for {nested_name}.") from None


def _resolve_inputs(input_root: Path | str) -> dict[str, _InputSource]:
    root = Path(input_root)
    matches: dict[str, list[_InputSource]] = {name: [] for name in INPUT_FIELDS}
    if root.is_file() and root.suffix.lower() == ".zip":
        _add_zip_sources(root, matches, direct=True)
    elif root.is_dir():
        for path in root.rglob("*"):
            if path.name in INPUT_FIELDS and path.is_file():
                matches[path.name].append(_InputSource(path))
            elif path.suffix.lower() == ".zip" and path.is_file():
                _add_zip_sources(path, matches, direct=False)
    else:
        raise FileNotFoundError("Select an accessible directory or ZIP containing the competition inputs.")

    missing = [name for name, sources in matches.items() if not sources]
    ambiguous = [name for name, sources in matches.items() if len(sources) > 1]
    missing_message = f"Missing required files: {', '.join(missing)}." if missing else ""
    if ambiguous:
        problem = f"Multiple copies found for {', '.join(ambiguous)}. Select an input with one copy of each."
        raise ValueError(f"{missing_message} {problem}".strip())
    if missing:
        raise FileNotFoundError(missing_message)
    return {name: sources[0] for name, sources in matches.items()}


def _read_content(source: _InputSource, name: str) -> bytes:
    try:
        if source.outer_member is None:
            with source.path.open("rb") as handle:
                content = handle.read(MAX_CSV_BYTES + 1)
            if len(content) > MAX_CSV_BYTES:
                raise ValueError(f"The input for {name} exceeds the supported read size.")
            return content
        with ZipFile(source.path) as archive:
            if source.inner_member is None:
                return _read_zip_member(archive, source.outer_member, MAX_CSV_BYTES, name)
            packed = _read_zip_member(archive, source.outer_member, MAX_NESTED_ZIP_BYTES, name)
            with ZipFile(BytesIO(packed)) as inner:
                return _read_zip_member(inner, source.inner_member, MAX_CSV_BYTES, name)
    except (BadZipFile, OSError, RuntimeError, NotImplementedError, DecompressionError):
        raise ValueError(f"Could not read the resolved competition input for {name}.") from None


def _read_input(source: _InputSource, name: str) -> pd.DataFrame:
    columns = list(INPUT_FIELDS[name])
    content = _read_content(source, name)
    try:
        frame = pd.read_csv(BytesIO(content), usecols=columns, dtype="string", keep_default_na=False)
    except (OSError, ValueError):
        raise ValueError(f"Expected a readable {name} with columns {', '.join(columns)}.") from None
    if frame.empty and name != "air_reserve.csv":
        raise ValueError(f"Expected at least one row in {name}.")
    return frame.loc[:, columns]


def _identifiers(values: pd.Series, field: str) -> pd.Series:
    identifiers = values.str.strip()
    if identifiers.isna().any() or identifiers.eq("").any():
        raise ValueError(f"Expected nonblank identifiers in {field}.")
    return identifiers


def _counts(values: pd.Series, field: str) -> pd.Series:
    numeric = pd.to_numeric(values, errors="coerce")
    if (
        numeric.isna().any()
        or not np.isfinite(numeric).all()
        or numeric.lt(0).any()
        or numeric.ge(2**63).any()
        or numeric.mod(1).ne(0).any()
    ):
        raise ValueError(f"Expected finite nonnegative integer counts in {field}.")
    return numeric.astype("int64")


def _dates(values: pd.Series, field: str, *, date_only: bool) -> pd.Series:
    date_format = "%Y-%m-%d" if date_only else "%Y-%m-%d %H:%M:%S"
    parsed = pd.to_datetime(values, format=date_format, errors="coerce")
    if parsed.isna().any():
        raise ValueError(f"Expected valid timezone-naive dates or timestamps in {field}.")
    return parsed.astype("datetime64[ns]")


def _scheduled_dates(sample: pd.DataFrame) -> pd.DatetimeIndex:
    ids = _identifiers(sample["id"], "sample_submission.csv id")
    if not ids.is_unique:
        raise ValueError("Expected unique IDs in sample_submission.csv.")
    if not ids.str.match(r"^.+_\d{4}-\d{2}-\d{2}$").all():
        raise ValueError("Expected sample_submission.csv IDs to end with _YYYY-MM-DD.")
    _counts(sample["visitors"], "sample_submission.csv visitors")
    date_suffix = _dates(ids.str[-10:], "sample_submission.csv ID date suffix", date_only=True)
    horizon = pd.DatetimeIndex(date_suffix.unique()).sort_values()
    if not horizon.equals(pd.date_range(horizon[0], periods=len(horizon), freq="D")):
        raise ValueError("The dates in sample_submission.csv must define a continuous daily horizon.")
    return horizon


def load_competition_data(
    input_root: Path | str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, int]:
    """Read a directory or ZIP and return validated data and the scheduled horizon length."""
    sources = _resolve_inputs(input_root)
    visits = _read_input(sources["air_visit_data.csv"], "air_visit_data.csv")
    visits["air_store_id"] = _identifiers(visits["air_store_id"], "air_visit_data.csv air_store_id")
    visits["visit_date"] = _dates(visits["visit_date"], "visit_date", date_only=True)
    visits["visitors"] = _counts(visits["visitors"], "air_visit_data.csv visitors")
    if visits.duplicated(subset=["air_store_id", "visit_date"]).any():
        raise ValueError("Expected one recorded visit count per AIR store and date.")

    calendar = _read_input(sources["date_info.csv"], "date_info.csv").rename(columns={"calendar_date": "visit_date"})
    calendar["visit_date"] = _dates(calendar["visit_date"], "calendar_date", date_only=True)
    calendar["holiday_flg"] = _counts(calendar["holiday_flg"], "holiday_flg")
    if not calendar["visit_date"].is_unique or not calendar["holiday_flg"].isin((0, 1)).all():
        raise ValueError("Expected one calendar row per date and holiday flags of 0 or 1.")

    reservations = _read_input(sources["air_reserve.csv"], "air_reserve.csv")
    reservations["air_store_id"] = _identifiers(reservations["air_store_id"], "air_reserve.csv air_store_id")
    for field in ("visit_datetime", "reserve_datetime"):
        reservations[field] = _dates(reservations[field], field, date_only=False)
    reservations["reserve_visitors"] = _counts(reservations["reserve_visitors"], "reserve_visitors")
    late = reservations["reserve_datetime"].gt(reservations["visit_datetime"])
    if late.any():
        LOGGER.warning("Ignored post-visit reservation records (count %d).", int(late.sum()))
        reservations = reservations.loc[~late].copy()
    reservations["visit_date"] = reservations["visit_datetime"].dt.normalize()

    horizon = _scheduled_dates(_read_input(sources["sample_submission.csv"], "sample_submission.csv"))
    if horizon[0] <= visits["visit_date"].max():
        raise ValueError("Expected the scheduled horizon to follow all recorded historical visits.")
    required_dates = pd.DatetimeIndex(visits["visit_date"].unique()).union(horizon)
    if not required_dates.difference(calendar["visit_date"]).empty:
        raise ValueError("Expected calendar coverage for all historical visit dates and the scheduled horizon.")

    return visits, calendar, reservations, len(horizon)

In [ ]:
"""Leakage-safe AIR visit backtests and aggregate planning scenarios."""

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from threadpoolctl import threadpool_limits

DAY = pd.Timedelta(days=1)
STORE_DATE = ["air_store_id", "visit_date"]
FEATURES = [
    "lead_days",
    "weekday",
    "month",
    "holiday_flg",
    "seasonal_naive",
    "store_mean",
    "store_history_days",
    "weekday_mean",
    "recent_28d_mean",
    "booked_reservations",
    "booked_visitors",
]
CANDIDATES = ("seasonal_naive", "hist_gradient_boosting")


def feature_snapshot(
    visits: pd.DataFrame,
    targets: pd.DataFrame,
    calendar: pd.DataFrame,
    reservations: pd.DataFrame,
    *,
    origin: pd.Timestamp,
) -> pd.DataFrame:
    """Create target-day features from a snapshot taken at the end of the origin date."""
    history = visits.loc[visits["visit_date"] <= origin].copy()
    if history.empty:
        raise ValueError("Forecast origins require earlier observed visits")

    features = targets.loc[:, STORE_DATE].copy()
    if (features["visit_date"] <= origin).any():
        raise ValueError("Target dates must be strictly after the forecast origin")
    features["lead_days"] = (features["visit_date"] - origin).dt.days
    features["weekday"] = features["visit_date"].dt.dayofweek
    features["month"] = features["visit_date"].dt.month
    features = features.merge(calendar[["visit_date", "holiday_flg"]], on="visit_date", how="left", validate="many_to_one")
    if features["holiday_flg"].isna().any():
        raise ValueError("Published calendar does not cover every target date")

    store = history.groupby("air_store_id").agg(
        store_mean=("visitors", "mean"),
        store_history_days=("visitors", "size"),
    )
    history["weekday"] = history["visit_date"].dt.dayofweek
    weekday = (
        history.sort_values("visit_date", kind="stable")
        .groupby(["air_store_id", "weekday"])
        .agg(weekday_mean=("visitors", "mean"), last_weekday_visits=("visitors", "last"))
    )
    recent = (
        history.loc[history["visit_date"] > origin - 28 * DAY]
        .groupby("air_store_id")["visitors"].mean().rename("recent_28d_mean")
    )
    features = features.join(store, on="air_store_id", validate="many_to_one")
    features = features.join(weekday, on=["air_store_id", "weekday"], validate="many_to_one")
    features = features.join(recent, on="air_store_id", validate="many_to_one")
    features["seasonal_naive"] = (
        features["last_weekday_visits"].fillna(features["store_mean"]).fillna(history["visitors"].mean())
    )
    features["store_history_days"] = features["store_history_days"].fillna(0)

    # Midnight after origin is excluded, even if the reservation precedes its visit.
    booked = reservations.loc[
        (reservations["reserve_datetime"] < origin + DAY)
        & (reservations["visit_date"] > origin)
        & reservations["visit_date"].isin(features["visit_date"])
    ]
    reservation_totals = booked.groupby(STORE_DATE).agg(
        booked_reservations=("reserve_visitors", "size"),
        booked_visitors=("reserve_visitors", "sum"),
    )
    features = features.join(reservation_totals, on=STORE_DATE, validate="many_to_one")
    features[["booked_reservations", "booked_visitors"]] = features[["booked_reservations", "booked_visitors"]].fillna(0)
    return features[FEATURES]


def target_window(visits: pd.DataFrame, origin: pd.Timestamp, horizon: pd.Timedelta) -> pd.DataFrame:
    """Select only observed store-days in the forecast horizon."""
    return visits.loc[(visits["visit_date"] > origin) & (visits["visit_date"] <= origin + horizon)]


def training_snapshots(
    visits: pd.DataFrame,
    calendar: pd.DataFrame,
    reservations: pd.DataFrame,
    *,
    origin: pd.Timestamp,
    horizon: pd.Timedelta,
) -> tuple[pd.DataFrame, np.ndarray, int]:
    """Replay consecutive, non-overlapping horizons with all labels known by origin."""
    oldest_visit = visits["visit_date"].min()
    historical_origins = []
    simulated_origin = origin - horizon
    while simulated_origin >= oldest_visit:
        historical_origins.append(simulated_origin)
        simulated_origin -= horizon

    predictors = []
    labels = []
    for simulated_origin in reversed(historical_origins):
        targets = target_window(visits, simulated_origin, horizon)
        if targets.empty:
            continue
        predictors.append(feature_snapshot(visits, targets, calendar, reservations, origin=simulated_origin))
        labels.append(targets["visitors"].to_numpy(dtype=float))

    if not predictors:
        raise ValueError("Not enough completed historical forecast windows to train the model")
    return pd.concat(predictors, ignore_index=True), np.concatenate(labels), len(predictors)


def error_totals(actual: np.ndarray, forecast: np.ndarray) -> tuple[float, float]:
    """Return additive absolute and squared log errors for exact pooling."""
    absolute = np.abs(forecast - actual).sum()
    squared_log = np.square(np.log1p(forecast) - np.log1p(actual)).sum()
    return float(absolute), float(squared_log)


def summarize_errors(totals: dict, n: int) -> dict:
    """Convert pooled loss totals to aggregate MAE and RMSLE."""
    return {
        "n_evaluated": n,
        "models": {
            key: {"mae": totals[key][0] / n, "rmsle": float(np.sqrt(totals[key][1] / n))}
            for key in CANDIDATES
        },
    }


def planning_scenario(actual: np.ndarray, forecast: np.ndarray, model: str) -> dict:
    """Estimate illustrative capacity with 18 visitors per shift and 10% headroom."""
    visitors_per_shift = 18
    headroom = 0.10
    without_headroom = np.ceil(forecast / visitors_per_shift).astype(np.int64)
    with_headroom = np.ceil(forecast * (1 + headroom) / visitors_per_shift).astype(np.int64)
    return {
        "model": model,
        "n_evaluated": len(actual),
        "visitors_per_shift": visitors_per_shift,
        "headroom": headroom,
        "shift_slots_without_headroom": int(without_headroom.sum()),
        "shift_slots_with_headroom": int(with_headroom.sum()),
        "under_capacity_days_without_headroom": int((actual > without_headroom * visitors_per_shift).sum()),
        "under_capacity_days_with_headroom": int((actual > with_headroom * visitors_per_shift).sum()),
    }


def evaluate_forecasts(
    visits: pd.DataFrame,
    calendar: pd.DataFrame,
    reservations: pd.DataFrame,
    *,
    horizon_days: int,
    n_folds: int = 3,
) -> dict:
    """Fit and test at rolling origins, returning only aggregate per-window results."""
    if not isinstance(horizon_days, (int, np.integer)) or horizon_days <= 0 or n_folds < 2:
        raise ValueError("A positive day horizon and at least two folds are required")
    if visits.empty:
        raise ValueError("Observed visits are required for the backtest")

    horizon = int(horizon_days) * DAY
    latest_date = visits["visit_date"].max()
    folds = []
    totals = {key: np.zeros(2) for key in CANDIDATES}
    n_overall = 0
    for index in range(n_folds):
        holdout = index == n_folds - 1
        if holdout:
            validation = summarize_errors(totals, n_overall)
            selected = min(CANDIDATES, key=lambda key: validation["models"][key]["rmsle"])

        origin = latest_date - (n_folds - index) * horizon
        targets = target_window(visits, origin, horizon)
        if targets.empty:
            raise ValueError(f"No target observations in window after {origin.date()}")
        train, labels, n_simulated_origins = training_snapshots(
            visits, calendar, reservations, origin=origin, horizon=horizon,
        )
        future = feature_snapshot(visits, targets, calendar, reservations, origin=origin)
        model = HistGradientBoostingRegressor(
            max_iter=150,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            learning_rate=0.07,
            l2_regularization=1,
            early_stopping=False,
            random_state=42,
        )
        with threadpool_limits(limits=1):
            model.fit(train, labels)
            prediction = np.maximum(model.predict(future), 0)
        forecasts = {"seasonal_naive": future["seasonal_naive"].to_numpy(dtype=float), "hist_gradient_boosting": prediction}
        actual = targets["visitors"].to_numpy(dtype=float)
        losses = {key: error_totals(actual, forecast) for key, forecast in forecasts.items()}
        folds.append({
            "name": "holdout" if holdout else f"validation_{index + 1}",
            "origin": origin.date().isoformat(),
            "target_start": (origin + DAY).date().isoformat(),
            "target_end": (origin + horizon).date().isoformat(),
            "n_train": len(labels),
            "n_simulated_training_origins": n_simulated_origins,
            **summarize_errors(losses, len(actual)),
        })
        for key in CANDIDATES:
            totals[key] += losses[key]
        n_overall += len(actual)

        if holdout:
            planning = planning_scenario(actual, forecasts[selected], selected)

    return {
        "horizon_days": int(horizon_days),
        "n_folds": n_folds,
        "folds": folds,
        "validation": validation,
        "overall": summarize_errors(totals, n_overall),
        "selection": {"model": selected, "criterion": "pooled_validation_rmsle", "tie_breaker": "seasonal_naive"},
        "planning": planning,
    }

In [ ]:
"""Export only aggregate forecast metrics and charts for review."""

import json
from io import StringIO
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZIP_DEFLATED, ZipFile, ZipInfo

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

MODEL_LABELS = {
    "seasonal_naive": "Seasonal naïve",
    "hist_gradient_boosting": "Gradient boosting",
}


def plot_fold_errors(folds, metric, ylabel, destination):
    """Chart the same aggregate error metric for both models on each window."""
    positions = np.arange(len(folds))
    fig, ax = plt.subplots(figsize=(6.4, 3.5), layout="constrained")
    try:
        for offset, (model, label) in enumerate(MODEL_LABELS.items()):
            values = [fold["models"][model][metric] for fold in folds]
            ax.bar(
                positions + (offset - 0.5) * 0.34,
                values,
                width=0.34,
                label=label,
                color=("#527b9c", "#176d69")[offset],
            )
        ax.set_xticks(positions, [fold["name"].replace("_", " ").title() for fold in folds])
        ax.set_ylabel(ylabel)
        ax.set_ylim(bottom=0)
        ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", alpha=0.2)
        ax.set_axisbelow(True)
        with StringIO() as svg, matplotlib.rc_context({"svg.hashsalt": "restaurant-forecasting"}):
            fig.savefig(svg, format="svg", metadata={"Title": ylabel, "Date": None})
            destination.write_text(
                "\n".join(line.rstrip() for line in svg.getvalue().splitlines()) + "\n",
                encoding="utf-8",
            )
    finally:
        plt.close(fig)


def write_report(results, output_dir: Path) -> Path:
    """Write a small archive of aggregate results, with an explicit file allowlist."""
    output_dir.mkdir(parents=True, exist_ok=True)
    archive_path = output_dir / "restaurant-forecasting-results.zip"
    with TemporaryDirectory(dir=output_dir) as temp:
        stage = Path(temp)
        (stage / "metrics.json").write_text(
            json.dumps(results, indent=2, sort_keys=True, allow_nan=False) + "\n",
            encoding="utf-8",
        )
        files = ["metrics.json"]
        for metric, ylabel in (("mae", "Mean absolute error (visitors)"), ("rmsle", "RMSLE")):
            filename = f"{metric}-by-window.svg"
            plot_fold_errors(results["folds"], metric, ylabel, stage / filename)
            files.append(filename)
        temporary_archive = stage / archive_path.name
        with ZipFile(temporary_archive, "w", ZIP_DEFLATED) as archive:
            for filename in files:
                archive.writestr(ZipInfo(filename), (stage / filename).read_bytes(), compress_type=ZIP_DEFLATED)
        temporary_archive.replace(archive_path)
    return archive_path

In [ ]:
import os

# Optional local overrides accept a directory or ZIP in an eligible workspace.
input_override = os.getenv("RESTAURANT_FORECAST_INPUT_DIR")
output_override = os.getenv("RESTAURANT_FORECAST_OUTPUT_DIR")
input_root = Path(input_override or "/kaggle/input")

try:
    visits, calendar, reservations, horizon_days = load_competition_data(input_root)
except FileNotFoundError:
    if input_override or not Path("/kaggle/working").is_dir():
        raise
    import kagglehub

    downloaded = kagglehub.competition_download("recruit-restaurant-visitor-forecasting")
    visits, calendar, reservations, horizon_days = load_competition_data(downloaded)

forecast_results = evaluate_forecasts(visits, calendar, reservations, horizon_days=horizon_days)
output_directory = Path(output_override or "/kaggle/working")
results_archive = write_report(forecast_results, output_directory)

for fold in forecast_results["folds"]:
    print(fold["name"], fold["models"])
print("Aggregate results archive:", results_archive)